In [1]:
# basic
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
# pre processing
from sklearn import preprocessing as pre
# NN
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import torch.optim as optim
from torch.nn import MSELoss
from torch_geometric.nn import GCNConv
# val and plot
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from loguru import logger as log
#from ..val import calculate_metrics
# plot
import matplotlib.pyplot as plt
# foundation model
from functools import reduce

/home/marcos/.pyenv/versions/3.10.13/envs/gnn-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import itertools
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [3]:
import pmdarima as pm

In [4]:
plt.style.use("seaborn-v0_8-whitegrid")

In [5]:
SEED = 1345
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
#plt.style.use('seaborn-whitegrid')
#pd.set_option('display.float_format', '{:.16f}'.format)
warnings.filterwarnings('ignore')

In [6]:
def load_datasets(filepath):
    """Carrega os datasets de arquivos pickle."""
    try:
        with open(filepath, 'rb') as f:
            dataset = pickle.load(f)
        return dataset
    except IOError as e:
        log.error(f"Erro ao carregar o dataset: {e}")
    except pickle.PickleError as e:
        log.error(f"Erro ao desserializar o dataset: {e}")
        traceback.print_exception(e)

## Data

In [7]:
sb = pd.read_parquet("/home/marcos/loader_03-04_2024.parquet")
sb.head()

,125960550,230565994,258781031,43768720,44072192,44783654,44783914,44784438,45833547,47568123
2024-03-01 05:00:00,1.333333,0.000000,70.792221,41.188599,1.724359,14.955100,2.032506,3.571820,5.877792,7.546274
2024-03-01 05:30:00,3.595238,1.583333,229.051071,172.071198,8.282966,44.559937,11.048912,18.211931,18.912033,18.276293
2024-03-01 06:00:00,4.812975,3.268518,424.853729,433.062469,18.825665,97.263435,26.276600,41.471294,40.731876,37.141144
2024-03-01 06:30:00,9.215629,5.256614,630.444153,743.177368,25.593414,149.329544,49.763138,71.520836,57.200085,53.487366
2024-03-01 07:00:00,12.585028,6.152447,841.874512,1132.739502,44.350349,204.275940,78.721497,107.241295,77.808769,75.446609


In [8]:
sb.columns

Index(['125960550', '230565994', '258781031', '43768720', '44072192',
       '44783654', '44783914', '44784438', '45833547', '47568123'],
      dtype='object')

In [9]:
# define X and Y
sbx = sb.query("index <= '2024-03-31 23:59:59'")
sbx.shape, sb.shape

((1238, 10), (3640, 10))

In [10]:
# define X and Y
sby = sb.query("index > '2024-03-31 23:59:59'")
sby.shape, sb.shape

((2402, 10), (3640, 10))

In [11]:
pred_len = abs(sbx.shape[0] - sb.shape[0])
pred_len

2402

In [12]:
sby.shape

(2402, 10)

## Data in batch

In [13]:
fpath_root = "/mnt/data/marcos/data/node_regression_bus/data_split/"
test_dataset = load_datasets(f'{fpath_root}test.pkl')
# x (dados de input)
test_dataset[0].x.shape

torch.Size([2871, 280])

In [14]:
inference_size = test_dataset[0].y.shape[1]
inference_size

200

In [15]:
nodes = [53,  365,  382,  666,  701, 1326, 1404, 1569, 1916, 2617]

In [16]:
stops = {53: '125960550',
         365: '230565994',
         382: '258781031',
         666: '43768720',
         701: '44072192',
         1326: '44783654',
         1404: '44783914',
         1569: '44784438',
         1916: '45833547',
         2617: '47568123'}

### Forecasting

In [19]:
scores_error = {'node': [], 'batch': [], 'mae': [], 'mse': [], 'r2': [], 'mape': []}
targets = {}
cost, time = 0, 0
dfs = []

only_day = False

for node in nodes:
    dfs_pred = []    
    for time, snapshot in tqdm(enumerate(test_dataset)):
        snapshot.to('cpu')
        
        serie = sbx.copy()#[stops[node]]
        targets[node] = []

        #
        # Data
        #
        if only_day:
            serie_np = serie[stops[node]].values  # array tipo numpy
            nova_seq = snapshot.x[node, :].cpu().numpy()
            serie_expandida = np.concatenate([serie_np, nova_seq[-40:]])
        else:
            only_day = True
            serie_np = serie[stops[node]].values  # array tipo numpy
            nova_seq = snapshot.x[node, :].cpu().numpy()
            serie_expandida = np.concatenate([serie_np, nova_seq])
        
        #
        # (alterar aqui o modelo)
        #
        
        # ler os parametros do no
        with open(f"sarima_fit_gs/{stops[node]}-grid-search-results.pkl", "rb") as f:
            modelo_auto = pickle.load(f)
        
        modelo = SARIMAX(serie_expandida,
                         order=modelo_auto.order,
                         seasonal_order=modelo_auto.seasonal_order,
                         enforce_stationarity=False,
                         enforce_invertibility=False)

        resultado = modelo.fit()
        #
        y_true = snapshot.y[node, :].cpu().numpy()
        forecast = resultado.get_forecast(steps=y_true.shape[0])
        y_pred = forecast.predicted_mean
        
        scores_error['node'].append(stops[node])
        scores_error['batch'].append(time)
        scores_error['mse'].append(mean_squared_error(y_true, y_pred))
        scores_error['mae'].append(mean_absolute_error(y_true, y_pred))
        scores_error['r2'].append(r2_score(y_true, y_pred))
        scores_error['mape'].append(mean_absolute_percentage_error(y_true, y_pred))
        
        # targets.append({'true': y_true,
        #                 'pred': y_pred})
        targets[node].append({"input":  snapshot.x[node,:].cpu().numpy(), 
                              'true': y_true,
                              'pred': y_pred,
                              'node': stops[node]
                             })

38it [00:55,  1.45s/it]
38it [09:16, 14.65s/it]
38it [00:10,  3.73it/s]
38it [12:23, 19.56s/it]
38it [00:12,  3.00it/s]
38it [01:03,  1.68s/it]
38it [11:26, 18.06s/it]
38it [02:35,  4.08s/it]
38it [00:32,  1.16it/s]
38it [01:01,  1.63s/it]


In [20]:
model_name = "SARIMA-FIT"

In [21]:
with open(f'results/{model_name}-targets.pkl', 'wb') as f:
    pickle.dump(targets, f)

In [22]:
df_results = pd.DataFrame(scores_error)
df_results["model"] = model_name
df_results.to_parquet(f"results/{model_name}-batch.parquet", index=False)